# Very Very important SQL Transformation 

### DSL vs SQL

Anything is fine , performance wise no diffrenece , since all are going to run in low code (RDD)

-> both DSL + SQL
->  EL (DSL) + (Transformation + load ) SQL 

CREATE OR REPLACE TEMPORARY VIEW rawdf1view
(
  id INT,
  firstname STRING,
  lastname STRING,
  age INT,
  profession STRING
)
USING CSV
OPTIONS (
  path "/Volumes/workspace/default/volume1/custs",
  header "false",
  inferSchema "false" 
);

In [0]:
cust_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/custs",header=False,sep=",",inferSchema=True).toDF("custid","fname","lname","age","prof")

cust_df.show()

# transform cust usig sql transformation

 if prof null the fill with unknwon

 age_cat -> less than 18 minor , 18 to 59 -> major , above 59 sr citizen 

In [0]:
cust_df.createOrReplaceTempView("cust_tbl")

# spark.sql(sql_str)==> DF 

spark.sql("select * from cust_tbl").show()


In [0]:
spark.sql("select custid,fname,lname,age,nvl(prof,'unknown') as prof from cust_tbl").show()

In [0]:
# sql() = spark.sql()

final_df=spark.sql("select custid,fname,lname,case when age <18 then 'minor' when age <59 then 'major' else 'senior' end as age,nvl(prof,'unknown') as prof from cust_tbl")

final_df.show()

In [0]:
df=spark.range(10)

df.createOrReplaceTempView("tbl")

sql("select id, id*2 as new_id from tbl").show()

In [0]:
sql("show tables").show()

In [0]:
sql("select * from izwe48catalog.we48db.tbl_customer").show()

### Create a Dataframe using read (DSL)

In [0]:
cust_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/custs_header",header=True,sep=",",inferSchema=True)

cust_df.show()

cust_df.createOrReplaceTempView("tbl_cust_view")  # temp view / similar tem table / session table 

### select custid , fname,age 
### filter only Lawyer profession 

select custid,fname,age from tbl where profession='Lawyer'

In [0]:
flt_df=spark.sql("select custid,fname,age from tbl_cust_view where profession='Lawyer'") # DF 

flt_df.show()

flt_df.count()

### create view using pure SQL

In [0]:

# spark.sql(sql_qury)=> DF 
# notebook / REPL -> shotcut -> sql(query)=> DF



qry="""
create or replace temp view tbl_cust_view1 
using csv
options
(
    header=True,
    inferSchema=True,
    sep=",",
    path="/Volumes/izwe48catalog/we48db/staging/custs_header"
)
"""

# spark.sql(qry)
sql(qry)

In [0]:
print(spark.sql)
print(sql)

In [0]:
spark.sql("select * from tbl_cust_view1").show(5)

In [0]:
spark.sql("show tables in izwe48catalog.we48db").show()

In [0]:
qry="""
create or replace temp view tbl_new_cust
(
    custid int ,
    firstname string,
    lastname string,
    age int , 
    prof string
) 
using csv
options
(
    header=False, 
    sep=",",
    path="/Volumes/izwe48catalog/we48db/staging/custs"
)
"""

spark.sql(qry)

spark.sql("select * from tbl_new_cust").show(5)


DF => view - DSL(EL) 
ddl string (create or replace ...) => spark.sql()

In [0]:
%fs head /Volumes/izwe48catalog/we48db/staging/emp.json

In [0]:
# spark.read.json().createOrReplaceTempView()
# using json /csv / parquet / orc/ xml / delta / iceberg...
spark.sql("""
          create or replace temporary view emp_json
          using json
          options
          (
            path "/Volumes/izwe48catalog/we48db/staging/emp.json"
          )
          """)

In [0]:
spark.sql("select * from emp_json").show()

In [0]:
%sql

create or replace temporary view emp_json
          using json
          options
          (
            path "/Volumes/izwe48catalog/we48db/staging/emp.json"
          ) ;

select * from emp_json

In [0]:
# EDA -  describe , summary 

spark.sql("describe tbl_new_cust").show()

cust_df.printSchema()

cust_df.describe().show()
cust_df.summary().show()


In [0]:
%sql

select count(custid) as cust_count, min(custid) as min_custid, max(custid) as max_custid from tbl_new_cust;


-- metedata quries 

describe  izwe48catalog.we48db.tbl_cust_detail;
describe formatted izwe48catalog.we48db.tbl_cust_detail;

describe extended  izwe48catalog.we48db.tbl_cust_detail;



## Active Data munging 

1. Structuring - merging / Schema Evolution 
2. Cleansing , Scrubbing 
3. De duplication  

In [0]:
data1=[(100,"raja",32),(101,"sameer",42),(102,"ragav",32)]

data2=[(500,"krish",32),(502,"parthi",42),(503,"anton",42),(102,"ragav",32)]


df1=spark.createDataFrame(data1,["id","name","age"])
df2=spark.createDataFrame(data2,["id","name","age"])

display(df1)
display(df2)

df1.createOrReplaceTempView("emp1")
df2.createOrReplaceTempView("emp2")


# union 
# union all
# unionByNAme - not available in sql 



# union in dsl , combine return as it is (will not remove dulicate )-> union = unin all

# union in SQL , combine and  return unique records(dedup applied )

# union -> combine two table (similar structure) without duplicate 
# basd on the(poition + type) -> not based on the name 
spark.sql("select * from emp1 union select * from emp2").show()


# union all -> combine two table (similar structure) will not perform deduplication
print("union all")
spark.sql("select * from emp1 union all select * from emp2").show()

# unionByName -> we have to write our logic 

data3=[(1000,"chennai")]
df3=spark.createDataFrame(data3,["id","city"])

df3.createOrReplaceTempView("emp3")

spark.sql("select id,name,age, null as city from emp1 union select id,null as name , null as age , city from emp3").show()



## null handling - cleansing , scrubbing 


na functions - na.drop / na.fill / na.replace 

In [0]:
data=[
    (100,"crish",25),
    (101,"bala",None),
    (102,None,None),
    (None,None,None),
    (103,"raja",None),
    (104,None,25),
]


df=spark.createDataFrame(data,["id","name","age"])

df.show()

df.createOrReplaceTempView("tbl_stud")

In [0]:
# dsl
# default -> any 

df.na.drop().show()
df.na.drop(how="any").show()


qry="""
select * from tbl_stud where id is not null and name is not null and age is not null 
"""

spark.sql(qry).show()

In [0]:
df.show()
df.na.drop(how="all").show()  # dropping if all columns are null 


qry="""
select * from tbl_stud where id is not null or name is not null or age is not null 
"""
# taking row if  any column has value 
spark.sql(qry).show()

In [0]:
## na fill - scrubbing 

df.show()

df.na.fill(0).na.fill("NA").show()

In [0]:
spark.sql("select * from tbl_stud").show()

# if id is null -> 0 , if age is null -1 , if name is null -> NA

# case when 

spark.sql("select case when id is null then 0 else id end  as id,nvl(name,'NA') as name,nvl(age,-1) as age  from tbl_stud").show()

# nvl

spark.sql("select nvl(id,0) as id,nvl(name,'NA') as name,nvl(age,-1) as age  from tbl_stud").show()

# coalesce 

spark.sql("select coalesce(id,0) as id,coalesce(name,'NA') as name,coalesce(age,-1) as age  from tbl_stud").show()







In [0]:
%sql


select nvl(null,null);
select coalesce(null,0,null,null,3,4,5);


--  case when id is null 'b' else 'a' end
select nvl2(100,'a','b');
select nvl2(null,'a','b') ;






select concat('vadi','iz',nvl(null,'')) ;;



select nvl(concat(nvl(fname,'inceptez)','_',nvl(lname,fname),'@iz.com'),'NA') from tbl_cust where custid<='4000004'


### adding column  / dropping colun / transforming a column / renaming a column  

DSL : withcolumn  , withColumnRename , select , selectExpr , drop

SQL  : select 

In [0]:
%fs head /Volumes/izwe48catalog/we48db/staging/BB2/custsmodified

In [0]:
ddl_str="custid int,fname string,lname string,age int,prof string"
cust_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/BB2/custsmodified",header=False,schema=ddl_str)
cust_df.show()
cust_df.createOrReplaceTempView("tbl_cust")

In [0]:
# custid,name,email,age,prof
# add a email column  -> fname+_+lanme@iz.com
# take only fname in intcap format - name as name
# prof is null defaulted self_emp
# if custid null put -1
# remove lname column
# age if null -1 
# dwh column - loaddt-currentdate, appuser(izuser)

# select nvl(concat(nvl(fname,'inceptez)','_',nvl(lname,fname),'@iz.com'),'NA') from tbl_cust where custid<='4000004'
qry="""
select nvl(custid,-1) as custid,
       initcap(fname) as name,
       nvl(concat(fname,'_',lname,'@iz.com'),'NA') as email ,
       nvl(age,-1) as age, 
       coalesce(prof,'self_emp') as prof,
       current_date() as load_dt,
       'izuser' as appuser
 from tbl_cust 
"""

spark.sql(qry).show()



### Handle duplicates 

group by having -> identify dups 


# removing duplicate 


disctinct ------> row level 

dropDuplicate   - both (without any arg like distinct ) + with arg column level as well 


rownumber  - priority based duplicate removal


In [0]:
%sql
-- record level deduplication 

select distinct * from tbl_cust;


-- check whether duplicate is there not 
-- if custid repease them consider duplicate 

select custid,count(1) from tbl_cust group by custid having count(1)>1


select  * from tbl_cust where custid in (4000003,4000001) or custid is null;

-- remove duplicate 

-- if custid has duplicate takes the latest (age higher)
-- drop all custid null records 


select custid,fname,lname,age,prof from 
(
select *,row_number() over (partition by custid order by age  desc) as rno
 from tbl_cust 
 where custid is not null
) where rno=1
;


-- using qualify 
-- avoiding the inline view for filtering 
-- adding rno col , removing from sel

select *
 from tbl_cust 
 qualify row_number() over (partition by custid order by age  desc)=1




In [0]:
spark.sql("select  * from tbl_cust").count()
spark.sql("select distinct * from tbl_cust").count()

In [0]:
# case when 

# age_cat -> if age >60 -> sr citizen
# age >45 -> mid age
# others -> young 

# DSL : when(cond,value).otherwise(val)

qry="""
select custid,fname,prof,age,
        case when age >60 then 'sr citizen'
             when age >45 then 'mid age'
             when age is null then 'NA' 
             else 'young'
        end as age_cat 
from tbl_cust

"""

spark.sql(qry).show()




In [0]:
data =[(100,"raja",23),(101,"abc",33),(100,"raja",23),(101,"abc",55),(500,"krish",23)]

df=spark.createDataFrame(data,["id","name","age"])
display(df)

df.createOrReplaceTempView("tbl_stud")


# record level - entire rec repeated

display(spark.sql("select distinct * from tbl_stud"))

display(spark.sql("select distinct id,name from tbl_stud"))

display(spark.sql("select id,count(*) as cnt from tbl_stud group by id"))



# check whther id has duplicate  ?
display(spark.sql("select id,count(*) as cnt from tbl_stud group by id having count(*)>1"))

# column level - consider id alone , if id repeats then its duplicate, take the latest age record

# row_number() over(partition by id order by age desc)

# inline sql 
row_num_qry="""
select id,name,age from (
select *,row_number() over(partition by id order by age desc) as rn from tbl_stud
) where rn=1
"""

display(spark.sql(row_num_qry))

# sql + DSL 

spark.sql("select *,row_number() over(partition by id order by age desc) as rn from tbl_stud").filter("rn=1").drop("rn").show()

# sql + DSL 
spark.sql("select *,row_number() over(partition by id order by age desc) as rn from tbl_stud").createOrReplaceTempView("tbl_row_stud")

display(spark.sql("select id,name,age from tbl_row_stud where rn=1"))

                         
                        

##  UDF - User defined function 


### DSL : 

step 1 : create / inmport your python function

Step 2 : import udf from pyspark.sql.functions 


step3 : convert python function into udf using udf functions 


### SQL : 

step 1 : create / inmport your python function

step2 : register python function into spark udf 


In [0]:
cust_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/custs",header=False,sep=",",inferSchema=True).toDF("custid","fname","lname","age","prof")

cust_df.createOrReplaceTempView("tbl_cust")


# age cat -try with udf 

def get_age_cat(age):
    if age >60:
        return "sr citizen"
    elif age >45:
        return "mid age"
    else:
        return "young" 

# test function

get_age_cat(55)

# register python finction into spark sql 

spark.udf.register("fn_get_agecat",get_age_cat) # scope in this session 


qry="""

select custid,fname,prof,age,
       fn_get_agecat(age) as age_cat from tbl_cust

"""
display(spark.sql(qry))


In [0]:
data=[(100,"raja",23),(101,"abc",55),(100,"raja",None)]

df=spark.createDataFrame(data,["id","name","age"])
display(df)
df.createOrReplaceTempView("tbl_test")

# create udf take age return multiple age with 2 using spark sql

def get_age_twice(age):
    if age is None:
        return None
    return age*2


spark.udf.register("age_twice",get_age_twice)

display(spark.sql("select id,name,age,age_twice(age) as age_cat from tbl_test"))


# udf using dsl 


def get_age_twice(age):
    if age is None:
        return None
    return age*2


from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType

age_twice_udf=udf(get_age_twice,IntegerType())

df.select("*",age_twice_udf("age")).show()



# lambda function with sql 

spark.udf.register("age_lambda",lambda age:age*2,"int") 

display(spark.sql("select id,name,age,age_lambda(nvl(age,0)) as age_cat from tbl_test"))



In [0]:
%sql

create function....

###  join 


1. inner
2. left
3. right
4. full
5. semi
6. anti


In [0]:
customer_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/joindata/c_customer.csv",sep="|",inferSchema=True,header=True)

trans_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/joindata/c_txns.csv",sep="|",inferSchema=True,header=True)

customer_df.createOrReplaceTempView("tbl_cust")

trans_df.createOrReplaceTempView("tbl_trans")

display(customer_df)
display(trans_df)

In [0]:
# inner / equi join 

# customer with trnasction detail (customer which has trans)
# cid,cname,amt,city

qry="""
select c.cid,c.cname,t.amt,c.city from tbl_cust c join tbl_trans t on c.cid=t.cid"""

display(spark.sql(qry))


In [0]:
# left join 

# all customer with matching  trnasction detail 
# cid,cname,amt,city

qry="""
select c.cid,c.cname,t.amt,c.city from tbl_cust c left join tbl_trans t on c.cid=t.cid"""

display(spark.sql(qry))


In [0]:
# right join 

# all transaction + matching customer info 
# tid ,cid,cname , city 
# cname -> unknown  , city -> NA

qry="""
select t.tid,t.cid,nvl(c.cname,'unknown') as cname,t.amt,nvl(c.city,'NA') as city from tbl_cust c right join tbl_trans t on c.cid=t.cid"""

display(spark.sql(qry))

In [0]:
# full join

# int is null place -1

# string null -> NA

qry="""
select * from tbl_cust c full join tbl_trans t on c.cid=t.cid"""

display(spark.sql(qry).na.fill(-1).na.fill("NA"))

In [0]:
# left semi 

# similar to inner but retun only left side table 

# simialr to exists 

# get the customer who has atleast 1 tansaction 

qry="""
select * from tbl_cust c left semi join tbl_trans t on c.cid=t.cid"""

display(spark.sql(qry))

In [0]:
# left Anti 



# simialr to not exists 

# get the customer who dont have any transaction 

qry="""
select * from tbl_cust c left anti join tbl_trans t on c.cid=t.cid"""

display(spark.sql(qry))

In [0]:
# exists syntax :(semi)


# corelated subquery

qry="""

select * from tbl_cust c where exists(select 1 from tbl_trans t where c.cid=t.cid)
"""

display(spark.sql(qry))

# in 

qry="""

select * from tbl_cust c where c.cid in  (select cid from tbl_trans)
"""

display(spark.sql(qry))

# not exists - anti join 

qry="""

select * from tbl_cust c where not exists(select 1 from tbl_trans t where c.cid=t.cid)
"""

display(spark.sql(qry))



# self Join 

### joining with same table 


In [0]:
data = [
    (101,"John",104,"IT"),
    (102,"Alice",104,"IT"),
    (103,"Bob",105,"HR"),
    (104,"David",106,"IT"),
    (105,"Emma",106,"HR"),
    (106,"James",None,"Management")
]

emp_df=spark.createDataFrame(data,["eid","name","mid","dept"])

emp_df.createOrReplaceTempView("tbl_emp")

display(emp_df)


In [0]:
# get all emp with manager name , if mnager not found - lvl1

# eid,name,manager_name,dept

# emp , manager  

# emp left manager on eid=mid

qry="""
select emp.eid,emp.name,nvl(mgr.name,'level1') as manager_name,emp.dept from tbl_emp emp 
 left join tbl_emp mgr on emp.mid=mgr.eid

"""

display(spark.sql(qry))

In [0]:
spark.sql("select * from izwe48catalog.we48db.tbl_cust_detail").show()

sql("describe formatted izwe48catalog.we48db.tbl_cust_detail").show()

display(sql("describe detail izwe48catalog.we48db.tbl_cust_detail"))

spark.read.table("izwe48catalog.we48db.tbl_cust_detail").show() # delta 

## group by + aggregations 

In [0]:
display(cust_df)

cust_df.createOrReplaceTempView("cust_df_view")

In [0]:
print(cust_df.columns)

display(spark.sql("select prof,count(1) as tot_rec from cust_df_view group by prof order by tot_rec desc"))


# taking prof which has more than 200 record , ordering the result based on the count desc
# prof,count,minage,maxage

display(spark.sql(""" 
                  select prof,count(1) as tot_rec,min(age) as agemin, max(age) as agemax 
                  from cust_df_view 
                  where prof is not null
                  group by prof 
                  having count(1)>200
                  order by tot_rec desc
                  """))


## windowing 

1.Rank

2.value / analytical

3.agg



In [0]:
# rank the  cust based on highest age on each prof


qry ="""
select * ,
row_number() over(partition by prof order by age desc) as rownumber,
rank() over(partition by prof order by age desc) as rank,
dense_rank() over(partition by prof order by age desc) as d_rank
from cust_df_view where prof in ('Pilot','Lawyer')
"""

display(spark.sql(qry))

In [0]:
data=[(100,"raja",45),(101,"abc",55),(102,"cde",None)]

df=spark.createDataFrame(data,["id","name","age"])

df.createOrReplaceTempView("df")  

sql("select *,row_number() over(order by age ) from df").show()

sql("select * from df order by age desc").show()


aggrgate functions with wwindow <br>
valued functions with window 

In [0]:
# rollup , cube
cust_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/custs_header",header=True,inferSchema=True)


sch_str="txnid long,txndt string,custid long,amt float,cat string,prod string,city string,state string,spendby string"
txn_df=spark.read.csv("/Volumes/izwe48catalog/we48db/staging/txns_2025.txt",schema=sch_str)

cust_txn_df=txn_df.join(cust_df,"custid")

cust_txn_df.show(5)


In [0]:
# Jersey City ,Oakland ,San Dieg

fl_df=cust_txn_df.filter("city in ('Jersey City','Oakland' ,'San Diego')")

fl_df.groupBy("city","spendby").sum("amt").show()

display(fl_df.rollup("city","spendby").sum("amt").orderBy("city"))

In [0]:
fl_df.createOrReplaceTempView("tbl_agg")

qry="""
select city,spendby,sum(amt) from tbl_agg group by rollup(city,spendby)
"""

display(sql(qry))